# Check Five Rows

Compare five feature rows with the source readings.

Open this notebook in `notebooks/M3/`. Run the cells in order. Saving again replaces this notebook's output files.


## Shared folder setup

In [1]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction')

## Choose the two inputs

In [2]:
from pathlib import Path
import pandas as pd
raw_path = PROJECT_ROOT / "data/pilot/half_hour_sample.csv"
feature_path = PROJECT_ROOT / "data/processed/prediction_inputs_sample.csv"
raw_path, feature_path

(WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/pilot/half_hour_sample.csv'),
 WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed/prediction_inputs_sample.csv'))

## Read the half-hour data

In [3]:
raw = pd.read_csv(raw_path, na_values=["Null", "null"])
raw["tstp"] = pd.to_datetime(raw["tstp"], utc=True)
raw["energy(kWh/hh)"] = pd.to_numeric(raw["energy(kWh/hh)"])
raw.head()

,LCLid,tstp,energy(kWh/hh)
0,MAC000414,2013-01-01 00:00:00+00:00,0.130
1,MAC000414,2013-01-01 00:30:00+00:00,0.125
2,MAC000414,2013-01-01 01:00:00+00:00,0.148
3,MAC000414,2013-01-01 01:30:00+00:00,0.095
4,MAC000414,2013-01-01 02:00:00+00:00,0.098


## Read the feature file

In [4]:
features = pd.read_csv(feature_path)
features["target_hour"] = pd.to_datetime(features["target_hour"], utc=True)
features = features.sort_values(["LCLid", "target_hour"])
features.head()

,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh,hour,weekday
0,MAC000414,2013-01-02 00:00:00+00:00,0.100,0.090,0.273,0,2
1,MAC000414,2013-01-02 01:00:00+00:00,0.121,0.100,0.193,1,2
2,MAC000414,2013-01-02 02:00:00+00:00,0.089,0.121,0.090,2,2
3,MAC000414,2013-01-02 03:00:00+00:00,0.116,0.089,0.091,3,2
4,MAC000414,2013-01-02 04:00:00+00:00,0.108,0.116,0.127,4,2


## See whether the keys are unique

In [5]:
raw.duplicated(["LCLid", "tstp"]).sum(), features.duplicated(["LCLid", "target_hour"]).sum()

(np.int64(0), np.int64(0))

## Choose the five kept examples

In [6]:
five_cases = [
    (1, "normal afternoon", "MAC000414", "2013-06-12 14:00"),
    (2, "usable after a gap-related exclusion", "MAC005456", "2013-01-26 10:00"),
    (3, "first possible kept hour", "MAC001312", "2013-01-02 00:00"),
    (4, "measured zero previous hour", "MAC000675", "2013-02-18 12:00"),
    (5, "midnight on clock-change date, UTC", "MAC000414", "2013-03-31 00:00")]
five_cases

[(1, 'normal afternoon', 'MAC000414', '2013-06-12 14:00'),
 (2, 'usable after a gap-related exclusion', 'MAC005456', '2013-01-26 10:00'),
 (3, 'first possible kept hour', 'MAC001312', '2013-01-02 00:00'),
 (4, 'measured zero previous hour', 'MAC000675', '2013-02-18 12:00'),
 (5, 'midnight on clock-change date, UTC', 'MAC000414', '2013-03-31 00:00')]

## Choose the four excluded examples

In [7]:
excluded_cases = [
    (6, "missing target", "MAC005456", "2013-08-18 05:00"),
    (7, "missing previous hour", "MAC005456", "2013-01-26 09:00"),
    (8, "missing yesterday", "MAC005456", "2013-01-27 08:00"),
    (9, "no earlier history", "MAC001312", "2013-01-01 00:00")]
excluded_cases

[(6, 'missing target', 'MAC005456', '2013-08-18 05:00'),
 (7, 'missing previous hour', 'MAC005456', '2013-01-26 09:00'),
 (8, 'missing yesterday', 'MAC005456', '2013-01-27 08:00'),
 (9, 'no earlier history', 'MAC001312', '2013-01-01 00:00')]

## Put the examples into one table

In [8]:
cases = pd.DataFrame(five_cases + excluded_cases, columns=["case_id", "case", "LCLid", "target_hour"])
cases["target_hour"] = pd.to_datetime(cases["target_hour"], utc=True)
cases["expected_kept"] = cases["case_id"] <= 5
cases = cases.set_index("case_id")
cases

,case,LCLid,target_hour,expected_kept
case_id,,,,
1,normal afternoon,MAC000414,2013-06-12 14:00:00+00:00,True
2,usable after a gap-related exclusion,MAC005456,2013-01-26 10:00:00+00:00,True
3,first possible kept hour,MAC001312,2013-01-02 00:00:00+00:00,True
4,measured zero previous hour,MAC000675,2013-02-18 12:00:00+00:00,True
5,"midnight on clock-change date, UTC",MAC000414,2013-03-31 00:00:00+00:00,True
6,missing target,MAC005456,2013-08-18 05:00:00+00:00,False
7,missing previous hour,MAC005456,2013-01-26 09:00:00+00:00,False
8,missing yesterday,MAC005456,2013-01-27 08:00:00+00:00,False
9,no earlier history,MAC001312,2013-01-01 00:00:00+00:00,False


## List the three hours needed per case

In [9]:
offsets = pd.DataFrame({"field": ["target_kwh", "lag_1h_kwh", "lag_24h_kwh"], "offset": [0, 1, 24]})
parts = cases.reset_index().merge(offsets, how="cross")
parts["hour_start"] = parts["target_hour"] - pd.to_timedelta(parts["offset"], unit="h")
parts[["case_id", "field", "hour_start"]].head(6)

,case_id,field,hour_start
0,1,target_kwh,2013-06-12 14:00:00+00:00
1,1,lag_1h_kwh,2013-06-12 13:00:00+00:00
2,1,lag_24h_kwh,2013-06-11 14:00:00+00:00
3,2,target_kwh,2013-01-26 10:00:00+00:00
4,2,lag_1h_kwh,2013-01-26 09:00:00+00:00
5,2,lag_24h_kwh,2013-01-25 10:00:00+00:00


## Find the correct half-hour end labels

In [10]:
parts["first_end"] = parts["hour_start"] + pd.Timedelta(minutes=30)
parts["second_end"] = parts["hour_start"] + pd.Timedelta(hours=1)
parts[["case_id", "field", "hour_start", "first_end", "second_end"]].head(3)

,case_id,field,hour_start,first_end,second_end
0,1,target_kwh,2013-06-12 14:00:00+00:00,2013-06-12 14:30:00+00:00,2013-06-12 15:00:00+00:00
1,1,lag_1h_kwh,2013-06-12 13:00:00+00:00,2013-06-12 13:30:00+00:00,2013-06-12 14:00:00+00:00
2,1,lag_24h_kwh,2013-06-11 14:00:00+00:00,2013-06-11 14:30:00+00:00,2013-06-11 15:00:00+00:00


## Index the raw readings

In [11]:
raw_values = raw.set_index(["LCLid", "tstp"])["energy(kWh/hh)"]
raw_values.head()

LCLid      tstp                     
MAC000414  2013-01-01 00:00:00+00:00    0.130
           2013-01-01 00:30:00+00:00    0.125
           2013-01-01 01:00:00+00:00    0.148
           2013-01-01 01:30:00+00:00    0.095
           2013-01-01 02:00:00+00:00    0.098
Name: energy(kWh/hh), dtype: float64

## Look up the first half-hour

In [12]:
first_keys = pd.MultiIndex.from_arrays([parts["LCLid"], parts["first_end"]])
parts["first_kwh"] = raw_values.reindex(first_keys).to_numpy()
parts[["case_id", "field", "first_kwh"]].head(3)

,case_id,field,first_kwh
0,1,target_kwh,0.236
1,1,lag_1h_kwh,0.313
2,1,lag_24h_kwh,0.103


## Look up the second half-hour

In [13]:
second_keys = pd.MultiIndex.from_arrays([parts["LCLid"], parts["second_end"]])
parts["second_kwh"] = raw_values.reindex(second_keys).to_numpy()
parts[["case_id", "field", "second_kwh"]].head(3)

,case_id,field,second_kwh
0,1,target_kwh,0.182
1,1,lag_1h_kwh,0.197
2,1,lag_24h_kwh,0.067


## Independently add the two readings

In [14]:
parts["recomputed_kwh"] = parts["first_kwh"] + parts["second_kwh"]
parts[["case_id", "field", "hour_start", "first_end",
    "second_end", "first_kwh", "second_kwh", "recomputed_kwh"]]

,case_id,field,hour_start,first_end,second_end,first_kwh,second_kwh,recomputed_kwh
0,1,target_kwh,2013-06-12 14:00:00+00:00,2013-06-12 14:30:00+00:00,2013-06-12 15:00:00+00:00,0.236,0.182,0.418
1,1,lag_1h_kwh,2013-06-12 13:00:00+00:00,2013-06-12 13:30:00+00:00,2013-06-12 14:00:00+00:00,0.313,0.197,0.510
2,1,lag_24h_kwh,2013-06-11 14:00:00+00:00,2013-06-11 14:30:00+00:00,2013-06-11 15:00:00+00:00,0.103,0.067,0.170
3,2,target_kwh,2013-01-26 10:00:00+00:00,2013-01-26 10:30:00+00:00,2013-01-26 11:00:00+00:00,0.216,0.153,0.369
4,2,lag_1h_kwh,2013-01-26 09:00:00+00:00,2013-01-26 09:30:00+00:00,2013-01-26 10:00:00+00:00,0.227,0.227,0.454
5,2,lag_24h_kwh,2013-01-25 10:00:00+00:00,2013-01-25 10:30:00+00:00,2013-01-25 11:00:00+00:00,0.120,0.066,0.186
6,3,target_kwh,2013-01-02 00:00:00+00:00,2013-01-02 00:30:00+00:00,2013-01-02 01:00:00+00:00,0.039,0.018,0.057
7,3,lag_1h_kwh,2013-01-01 23:00:00+00:00,2013-01-01 23:30:00+00:00,2013-01-02 00:00:00+00:00,0.042,0.038,0.080
8,3,lag_24h_kwh,2013-01-01 00:00:00+00:00,2013-01-01 00:30:00+00:00,2013-01-01 01:00:00+00:00,0.018,0.041,0.059
9,4,target_kwh,2013-02-18 12:00:00+00:00,2013-02-18 12:30:00+00:00,2013-02-18 13:00:00+00:00,0.076,0.000,0.076


## Arrange the independently calculated values

In [15]:
energy_fields = ["target_kwh", "lag_1h_kwh", "lag_24h_kwh"]
recomputed = parts.pivot(index="case_id", columns="field", values="recomputed_kwh")
recomputed = recomputed[energy_fields]
recomputed

field,target_kwh,lag_1h_kwh,lag_24h_kwh
case_id,,,
1,0.418,0.510,0.170
2,0.369,0.454,0.186
3,0.057,0.080,0.059
4,0.076,0.000,0.000
5,0.716,0.968,0.741
6,NaN,NaN,NaN
7,0.454,NaN,0.397
8,0.365,0.201,NaN
9,0.059,NaN,NaN


## Look up the submitted feature values

In [16]:
feature_values = features.set_index(["LCLid", "target_hour"])
case_keys = pd.MultiIndex.from_arrays([cases["LCLid"], cases["target_hour"]])
actual = feature_values.reindex(case_keys)
actual.index = cases.index
actual

,target_kwh,lag_1h_kwh,lag_24h_kwh,hour,weekday
case_id,,,,,
1,0.418,0.510,0.170,14.0,2.0
2,0.369,0.454,0.186,10.0,5.0
3,0.057,0.080,0.059,0.0,2.0
4,0.076,0.000,0.000,12.0,0.0
5,0.716,0.968,0.741,0.0,6.0
6,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN


## Check which examples are present

In [17]:
present = pd.Series(case_keys.isin(feature_values.index), index=cases.index, name="present_in_features")
membership = cases[["case", "expected_kept"]].join(present)
membership["matches_expectation"] = membership["expected_kept"] == membership["present_in_features"]
membership

,case,expected_kept,present_in_features,matches_expectation
case_id,,,,
1,normal afternoon,True,True,True
2,usable after a gap-related exclusion,True,True,True
3,first possible kept hour,True,True,True
4,measured zero previous hour,True,True,True
5,"midnight on clock-change date, UTC",True,True,True
6,missing target,False,False,True
7,missing previous hour,False,False,True
8,missing yesterday,False,False,True
9,no earlier history,False,False,True


## Compare the three energy values

In [18]:
energy_matches = (actual[energy_fields] - recomputed).abs() < 0.0005
energy_matches.loc[1:5]

,target_kwh,lag_1h_kwh,lag_24h_kwh
case_id,,,
1,True,True,True
2,True,True,True
3,True,True,True
4,True,True,True
5,True,True,True


## Calculate calendar expectations

In [19]:
calendar = pd.DataFrame(index=cases.index)
calendar["hour"] = cases["target_hour"].dt.hour
calendar["weekday"] = cases["target_hour"].dt.dayofweek
calendar["weekday_name"] = cases["target_hour"].dt.day_name()
calendar

C:\Users\Mystery\AppData\Local\Temp\ipykernel_54524\672864891.py:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  calendar["weekday"] = cases["target_hour"].dt.dayofweek


,hour,weekday,weekday_name
case_id,,,
1,14,2,Wednesday
2,10,5,Saturday
3,0,2,Wednesday
4,12,0,Monday
5,0,6,Sunday
6,5,6,Sunday
7,9,5,Saturday
8,8,6,Sunday
9,0,1,Tuesday


## Compare the two calendar fields

In [20]:
calendar_matches = actual[["hour", "weekday"]] == calendar[["hour", "weekday"]]
calendar_matches.loc[1:5]

,hour,weekday
case_id,,
1,True,True
2,True,True
3,True,True
4,True,True
5,True,True


## Collect the five real-row results

In [21]:
checked = cases.loc[1:5].join(actual.loc[1:5])
checked = checked.join(recomputed.loc[1:5].add_prefix("recomputed_"))
checked = checked.join(calendar[["weekday_name"]])
checked["all_match"] = present.loc[1:5] & energy_matches.loc[1:5].all(axis=1) & calendar_matches.loc[1:5].all(axis=1)
checked

,case,LCLid,target_hour,expected_kept,target_kwh,lag_1h_kwh,lag_24h_kwh,hour,weekday,recomputed_target_kwh,recomputed_lag_1h_kwh,recomputed_lag_24h_kwh,weekday_name,all_match
case_id,,,,,,,,,,,,,,
1,normal afternoon,MAC000414,2013-06-12 14:00:00+00:00,True,0.418,0.510,0.170,14.0,2.0,0.418,0.510,0.170,Wednesday,True
2,usable after a gap-related exclusion,MAC005456,2013-01-26 10:00:00+00:00,True,0.369,0.454,0.186,10.0,5.0,0.369,0.454,0.186,Saturday,True
3,first possible kept hour,MAC001312,2013-01-02 00:00:00+00:00,True,0.057,0.080,0.059,0.0,2.0,0.057,0.080,0.059,Wednesday,True
4,measured zero previous hour,MAC000675,2013-02-18 12:00:00+00:00,True,0.076,0.000,0.000,12.0,0.0,0.076,0.000,0.000,Monday,True
5,"midnight on clock-change date, UTC",MAC000414,2013-03-31 00:00:00+00:00,True,0.716,0.968,0.741,0.0,6.0,0.716,0.968,0.741,Sunday,True


## Keep the raw calculation evidence

In [22]:
raw_evidence = parts.pivot(index="case_id",
    columns="field", values=["first_kwh", "second_kwh",
    "hour_start", "first_end", "second_end"])
raw_evidence.columns = [field + "_" + kind for kind, field in raw_evidence.columns]
checked = checked.join(raw_evidence)
checked.head(1)

,case,LCLid,target_hour,expected_kept,target_kwh,lag_1h_kwh,lag_24h_kwh,hour,weekday,recomputed_target_kwh,...,target_kwh_second_kwh,lag_1h_kwh_hour_start,lag_24h_kwh_hour_start,target_kwh_hour_start,lag_1h_kwh_first_end,lag_24h_kwh_first_end,target_kwh_first_end,lag_1h_kwh_second_end,lag_24h_kwh_second_end,target_kwh_second_end
case_id,,,,,,,,,,,,,,,,,,,,,
1,normal afternoon,MAC000414,2013-06-12 14:00:00+00:00,True,0.418,0.51,0.17,14.0,2.0,0.418,...,0.182,2013-06-12 13:00:00+00:00,2013-06-11 14:00:00+00:00,2013-06-12 14:00:00+00:00,2013-06-12 13:30:00+00:00,2013-06-11 14:30:00+00:00,2013-06-12 14:30:00+00:00,2013-06-12 14:00:00+00:00,2013-06-11 15:00:00+00:00,2013-06-12 15:00:00+00:00


## What a plain row shift would use

In [23]:
shifted = features[["LCLid", "target_hour"]].copy()
by_home = features.groupby("LCLid")
shifted["previous_row_time"] = by_home["target_hour"].shift(1)
shifted["row_24_back_time"] = by_home["target_hour"].shift(24)
shifted["if_row_shift_lag_1h"] = by_home["target_kwh"].shift(1)
shifted["if_row_shift_lag_24h"] = by_home["target_kwh"].shift(24)
shifted.head()

,LCLid,target_hour,previous_row_time,row_24_back_time,if_row_shift_lag_1h,if_row_shift_lag_24h
0,MAC000414,2013-01-02 00:00:00+00:00,NaT,NaT,NaN,NaN
1,MAC000414,2013-01-02 01:00:00+00:00,2013-01-02 00:00:00+00:00,NaT,0.100,NaN
2,MAC000414,2013-01-02 02:00:00+00:00,2013-01-02 01:00:00+00:00,NaT,0.121,NaN
3,MAC000414,2013-01-02 03:00:00+00:00,2013-01-02 02:00:00+00:00,NaT,0.089,NaN
4,MAC000414,2013-01-02 04:00:00+00:00,2013-01-02 03:00:00+00:00,NaT,0.116,NaN


## Compare row shifts with exact times

In [24]:
shift_examples = shifted.set_index(["LCLid", "target_hour"]).reindex(case_keys[:5])
shift_examples.index = cases.loc[1:5].index
shift_examples["required_lag_1h_time"] = cases.loc[1:5, "target_hour"] - pd.Timedelta(hours=1)
shift_examples["required_lag_24h_time"] = cases.loc[1:5, "target_hour"] - pd.Timedelta(hours=24)
checked = checked.join(shift_examples)
shift_examples

,previous_row_time,row_24_back_time,if_row_shift_lag_1h,if_row_shift_lag_24h,required_lag_1h_time,required_lag_24h_time
case_id,,,,,,
1,2013-06-12 13:00:00+00:00,2013-06-11 14:00:00+00:00,0.510,0.170,2013-06-12 13:00:00+00:00,2013-06-11 14:00:00+00:00
2,2013-01-26 07:00:00+00:00,2013-01-25 08:00:00+00:00,0.351,0.235,2013-01-26 09:00:00+00:00,2013-01-25 10:00:00+00:00
3,NaT,NaT,NaN,NaN,2013-01-01 23:00:00+00:00,2013-01-01 00:00:00+00:00
4,2013-02-18 11:00:00+00:00,2013-02-17 12:00:00+00:00,0.000,0.000,2013-02-18 11:00:00+00:00,2013-02-17 12:00:00+00:00
5,2013-03-30 23:00:00+00:00,2013-03-30 00:00:00+00:00,0.968,0.741,2013-03-30 23:00:00+00:00,2013-03-30 00:00:00+00:00


## Find raw-data reasons for exclusion

In [25]:
missing = recomputed.isna()
missing.loc[6:9]

field,target_kwh,lag_1h_kwh,lag_24h_kwh
case_id,,,
6,True,True,True
7,False,True,False
8,False,False,True
9,False,True,True


## Assign the same priority as the builder

In [26]:
reason = pd.Series("kept", index=cases.index)
reason[missing["lag_24h_kwh"]] = "missing_lag_24h"
reason[missing["lag_1h_kwh"]] = "missing_lag_1h"
reason[missing["target_kwh"]] = "missing_target"
reason.loc[6:9]

case_id
6     missing_target
7     missing_lag_1h
8    missing_lag_24h
9     missing_lag_1h
dtype: str

## Check the specific expected reasons

In [27]:
expected_reason = pd.Series(["missing_target",
    "missing_lag_1h", "missing_lag_24h", "missing_lag_1h"],
    index=cases.loc[6:9].index)
excluded = cases.loc[6:9].join(recomputed.add_prefix("recomputed_"))
excluded = excluded.join(missing.add_prefix("missing_")).join(present)
excluded["reason"] = reason
excluded["expected_reason"] = expected_reason
excluded["as_expected"] = ~present.loc[6:9] & missing.loc[6:9].any(axis=1) & (reason.loc[6:9] == expected_reason)
excluded

,case,LCLid,target_hour,expected_kept,recomputed_target_kwh,recomputed_lag_1h_kwh,recomputed_lag_24h_kwh,missing_target_kwh,missing_lag_1h_kwh,missing_lag_24h_kwh,present_in_features,reason,expected_reason,as_expected
case_id,,,,,,,,,,,,,,
6,missing target,MAC005456,2013-08-18 05:00:00+00:00,False,NaN,NaN,NaN,True,True,True,False,missing_target,missing_target,True
7,missing previous hour,MAC005456,2013-01-26 09:00:00+00:00,False,0.454,NaN,0.397,False,True,False,False,missing_lag_1h,missing_lag_1h,True
8,missing yesterday,MAC005456,2013-01-27 08:00:00+00:00,False,0.365,0.201,NaN,False,False,True,False,missing_lag_24h,missing_lag_24h,True
9,no earlier history,MAC001312,2013-01-01 00:00:00+00:00,False,0.059,NaN,NaN,False,True,True,False,missing_lag_1h,missing_lag_1h,True


## Attach evidence to excluded examples

In [28]:
excluded = excluded.join(raw_evidence)
excluded[["case", "reason", "present_in_features", "as_expected"]]

,case,reason,present_in_features,as_expected
case_id,,,,
6,missing target,missing_target,False,True
7,missing previous hour,missing_lag_1h,False,True
8,missing yesterday,missing_lag_24h,False,True
9,no earlier history,missing_lag_1h,False,True


## Summary

In [29]:
checked["all_match"].all(), excluded["as_expected"].all()

(np.True_, np.True_)

## Choose the output folder

In [30]:
output_dir = PROJECT_ROOT / 'data/processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed')

## Save the five checked rows

In [31]:
five_path = output_dir / "five_row_check.csv"
checked.to_csv(five_path, index_label="case_id")
five_path

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed/five_row_check.csv')

## Save the excluded examples

In [32]:
excluded_path = output_dir / "missing_input_examples.csv"
excluded.to_csv(excluded_path, index_label="case_id")
excluded_path

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed/missing_input_examples.csv')